# AII 600. Lab 6: one point per precinct

Due before class, 7 October 2026.

Upload this notebook on Canvas, with every code cell executed and the output left in. Credit is on-time submission. I do not mark the lab.

Python kernel. R is allowed: switch to an R kernel and rewrite the code cells. Ask the tutor to translate `check`.

On 24 September 2026, [Meduza plotted the Duma party-list results](https://meduza.io/feature/2026/09/24/vybory-v-gosdumu-2026-samye-gryaznye-parlamentskie-vybory-epohi-putina): one point per precinct, turnout on $x$, United Russia share of valid votes on $y$. They did not publish the drawing code or the 82,124-row extract from the evening of 23 September (Leningrad oblast and the occupied regions were still missing). You will not get a pixel-identical figure, and you should not try.

You have a later freeze: 87,799 precincts with a party-list count, about 99% of 88,807, from the CEC protocols as compiled at [deg.zhizhin.xyz](https://deg.zhizhin.xyz/shpilkin.html) on 24 September. The right-hand mass will look a bit thicker than Meduza's. The 2003--2021 tables are the Shpilkin extracts in [Dmitry Kobak's elections repo](https://github.com/dkobak/elections).

| Ex. | Question | Tool |
|---|---|---|
| 1 | What does the 2026 scatter look like, and does extra turnout come with extra United Russia? | one point per precinct; OLS slope |
| 2 | What is the Moscow column? | subgroup, `is_deg` |
| 3 | Did the 2021 comet nucleus move? | 2021 vs 2026, a turnout band |
| 4 | How did the cloud change from 2003 to 2026? | six-election sequence, correlation |
| 5 | How many precincts sit on a round percentage? | integer peaks, a stated rule |

Work each numeric question on paper first, then type the expression. After you finish, copy whatever you cannot write from memory onto your cheat sheet.


## How this notebook works

1. Run the next cell once. It defines `check` and `plot_comet`.
2. Each exercise has an answer cell and a test cell. In the answer cell, replace every `None` and run it. Then run the test cell.
3. The test cell prints `ok  name` when that value is right to six decimals, `name is not right` when it is not (it never shows the target), and `replace None for name` when you left a `None`. Leave the output in the notebook.
4. Written cells have no test. Replace YOUR ANSWER HERE, then ask your tutor whether the reasoning holds. Prediction cells are for you: write them before you run the code, and do not go back and fix them.

The two CSV files sit next to this notebook. `pandas.read_csv` reads `.csv.gz` without extra steps. Do not download a fresh file from zhizhin: that dump refreshes, and the hashes here are for the freeze you were given.

Turnout is $x=$ ballots issued / registered voters. United Russia is $y=$ party-list votes / valid party-list votes. Plot them as percents, $100x$ and $100y$. City names are in Russian: `город Москва` is the city, not `Московская область`; `город Санкт-Петербург`; `Чеченская Республика`.

`numpy`, `matplotlib`, and `pandas` are allowed. Ask the tutor for syntax; do the reading of the plot yourself.


In [ ]:
from hashlib import sha256

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def check(name, value, expected, ndigits=6):
    if value is None:
        raise AssertionError(f"replace None for {name}")
    s = f"{round(float(value), ndigits):.{ndigits}f}"
    got = sha256(f"aii600-lab6|{name}|{s}".encode()).hexdigest()[:16]
    assert got == expected, f"{name} is not right"
    print("ok ", name)

def plot_comet(turnout, ur_share, ax=None, s=1, alpha=0.12, **kwargs):
    if ax is None:
        ax = plt.gca()
    ax.scatter(
        100 * np.asarray(turnout),
        100 * np.asarray(ur_share),
        s=s,
        alpha=alpha,
        linewidths=0,
        **kwargs,
    )
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.set_xlabel("Turnout (%)")
    ax.set_ylabel("United Russia, party list (%)")
    return ax


## Exercise 1: the 2026 cloud

Load `duma_2026.csv.gz`. Each row is one precinct. `issued` and `voters` give turnout; `ur_votes` and `valid` give United Russia's share of valid party-list votes. `is_deg` is 1 for remote-electronic totals (a handful of huge rows at 100% turnout), 0 for paper precincts.

**(a)** `n_2026` is the number of rows. `median_turnout` and `median_ur` are the unweighted precinct medians of the two shares (fractions, not percents).

**(b)** Ordinary least squares of `ur_share` on `turnout`, with intercept. `slope_ur_on_turnout` is the slope. If people who turn out were like people who stay home, extra turnout would not move the party share and the slope would be near 0. `np.polyfit(turnout, ur_share, 1)[0]` is the slope.

**(c)** Plot every precinct with `plot_comet`. You want a faint cloud, not a blob of ink: the helper already uses a small marker and low alpha.


In [ ]:
d26 = pd.read_csv("duma_2026.csv.gz")
d26["turnout"] = d26.issued / d26.voters
d26["ur_share"] = d26.ur_votes / d26.valid

n_2026 = None
median_turnout = None
median_ur = None
slope_ur_on_turnout = None

fig, ax = plt.subplots(figsize=(5, 4))
plot_comet(d26.turnout, d26.ur_share, ax)
ax.set_title("Duma 2026")


In [ ]:
check("n_2026", n_2026, "cf171f2c15544e1e")
check("median_turnout", median_turnout, "6a341b1456d2c113")
check("median_ur", median_ur, "332de14cb4be6a3f")
check("slope_ur_on_turnout", slope_ur_on_turnout, "b619119d28d48978")
print("ok  ex1")


**Write.** Look at the scatter, not the medians. Where is the dense part of the cloud, and what is the thick mass to the right? If the extra turnout on the right were people arriving independently of how they vote, what would you expect the slope in (b) to do? Two or three sentences.

YOUR ANSWER HERE


## Exercise 2: the Moscow column

Meduza's new mark on the 2026 plot is a vertical strip of precincts to the *left* of the main cloud. That is paper voting in the city of Moscow after most of the city voted remotely.

Filter `d26` to `region == "город Москва"` and `is_deg == 0`. `n_moscow_paper`, `moscow_median_turnout`, and `moscow_median_ur` are the count and the two medians (fractions). Then `spb_median_turnout` for `город Санкт-Петербург` (all rows) and `chechnya_median_ur` for `Чеченская Республика`.

Plot the rest of the country in grey and Moscow paper on top in another colour, same `plot_comet` scales.


In [ ]:
moscow_paper = None
n_moscow_paper = None
moscow_median_turnout = None
moscow_median_ur = None
spb_median_turnout = None
chechnya_median_ur = None

fig, ax = plt.subplots(figsize=(5, 4))
# rest of the country in grey, Moscow paper on top


In [ ]:
check("n_moscow_paper", n_moscow_paper, "f4cdaed68ce66c22")
check("moscow_median_turnout", moscow_median_turnout, "2ba7f9f0233b2c94")
check("moscow_median_ur", moscow_median_ur, "96e02f9092897263")
check("spb_median_turnout", spb_median_turnout, "b17946063a1ce773")
check("chechnya_median_ur", chechnya_median_ur, "e9e46bc565e5b851")
print("ok  ex2")


**Write.** (a) Why did Moscow paper turnout fall, and why is the remaining cloud a vertical strip rather than a diagonal tail? One of those has a mechanical explanation in the lab text; the shape of the strip does not. Two sentences.

(b) Petersburg and Chechnya moved differently from Moscow. What did each do, in one sentence each?

YOUR ANSWER HERE


## Exercise 3: 2021 vs 2026

Load `duma_hist.csv.gz`. It has `year`, `region`, `turnout`, `ur_share` as fractions for 2003, 2007, 2011, 2016, and 2021. Restrict to `year == 2021` for this exercise.

Meduza's reading of the 2021 plot is a comet: a nucleus around 35--40% turnout and 30% United Russia, and a tail that climbs with turnout. They treat the nucleus as the closest thing to an un-edited result, then note that this nucleus barely moved in 2026.

Take the nucleus to be precincts with turnout in $[0.30, 0.45]$, inclusive. `n_2021` is all 2021 rows. `n_nucleus_2021` and `n_nucleus_2026` are the counts in that band. `nucleus_ur_2021` and `nucleus_ur_2026` are the median United Russia shares inside it.

Side-by-side `plot_comet` for 2021 and 2026, same $x$ and $y$ limits.


**Predict**, before you run the cell. Will `nucleus_ur_2026` be within five percentage points of `nucleus_ur_2021`, or did the "honest" band jump with the official headline? One sentence.

YOUR PREDICTION HERE


In [ ]:
hist = pd.read_csv("duma_hist.csv.gz")
d21 = hist[hist.year == 2021]

n_2021 = None
n_nucleus_2021 = None
n_nucleus_2026 = None
nucleus_ur_2021 = None
nucleus_ur_2026 = None

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), sharex=True, sharey=True)
# 2021 on the left, 2026 on the right


In [ ]:
check("n_2021", n_2021, "b7f6451bba36bc21")
check("n_nucleus_2021", n_nucleus_2021, "ae00eab27dfd5c8e")
check("n_nucleus_2026", n_nucleus_2026, "76a80aa5ad98cd1c")
check("nucleus_ur_2021", nucleus_ur_2021, "4b8f89596cd0de2f")
check("nucleus_ur_2026", nucleus_ur_2026, "a20f0a18bb20652c")
print("ok  ex3")


**Write.** (a) How many points did the nucleus United Russia share move, and did the nucleus get smaller? Two sentences.

(b) Treating the nucleus as "true" is a modelling choice, not a theorem. Give one reason it could still be too high, and one reason the 2021--2026 comparison inside that band is still useful. Two sentences.

YOUR ANSWER HERE


## Exercise 4: six Duma elections

The last Meduza figure is the same scatter for every Duma election from 2003 through 2026. 2003 is still a compact cloud. Later years grow a tail, then a body.

Using `hist` and `d26`, compute Pearson's correlation of turnout with United Russia share in 2003, 2021, and 2026: `corr_2003`, `corr_2021`, `corr_2026`. Then the share of precincts with turnout above 0.8 in 2003 and in 2026: `share_turnout_gt80_2003`, `share_turnout_gt80_2026`.

Plot a $2\times 3$ grid, years 2003, 2007, 2011, 2016, 2021, 2026, each panel `plot_comet` with the same limits.


In [ ]:
corr_2003 = None
corr_2021 = None
corr_2026 = None
share_turnout_gt80_2003 = None
share_turnout_gt80_2026 = None

years = [2003, 2007, 2011, 2016, 2021, 2026]
fig, axes = plt.subplots(2, 3, figsize=(10, 6), sharex=True, sharey=True)
# one panel per year


In [ ]:
check("corr_2003", corr_2003, "2dff4d65250b74f4")
check("corr_2021", corr_2021, "51acfb37b49d4142")
check("corr_2026", corr_2026, "136e695e4eff6089")
check("share_turnout_gt80_2003", share_turnout_gt80_2003, "e648d8e1f5803099")
check("share_turnout_gt80_2026", share_turnout_gt80_2026, "659d4b6b4b64ec9a")
print("ok  ex4")


**Write.** A non-Gaussian cloud is not, by itself, evidence of anything: Canada 2011 was bimodal because English- and French-speaking regions voted differently. What is the argument that this sequence is not that story? Use 2003 and the fact that the country's geography did not jump. Two or three sentences.

YOUR ANSWER HERE


## Exercise 5: round percentages

One fingerprint that does not appear in honest counting is a pile of precincts at 50.0%, 60.0%, 70.0%. Kobak, Shpilkin, and Pshenichnikov ([*Significance*, 2018](https://rss.onlinelibrary.wiley.com/doi/10.1111/j.1740-9713.2018.01141.x); the rule is in their [*Annals* paper](https://projecteuclid.org/euclid.aoas/1458909907)) call a reported percentage $p$ on a 0--100 scale *near-integer* when $|p-\mathrm{round}(p)|\le 0.05$. That is a 0.1-point window around each integer.

Meduza's headline count, about 1,300 precincts in 2021 to about 2,250 in 2026, is the *excess* over a Monte Carlo baseline, on their incomplete 82,124-row dump, with their threshold. You will not match 2,250. Compute the raw count on this freeze, with the rule above.

A precinct is near-integer on turnout if that rule holds for $100\times$ turnout, and the same for United Russia. `n_int_turnout_2026`, `n_int_ur_2026`, and `n_int_both_2026` are the 2026 counts. `n_int_both_2021` is the 2021 count where *both* are near-integer.

Plot two histograms of 2026, 0.25-point bins from 0 to 100: turnout percent, then United Russia percent. You should see spikes at the round numbers. `np.arange(0, 100.25, 0.25)` is a set of edges.


In [ ]:
def near_integer(share, tol=0.05):
    p = 100 * np.asarray(share, float)
    return np.abs(p - np.round(p)) <= tol

n_int_turnout_2026 = None
n_int_ur_2026 = None
n_int_both_2026 = None
n_int_both_2021 = None

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
# turnout on top, United Russia below, bins of 0.25


In [ ]:
check("n_int_turnout_2026", n_int_turnout_2026, "e9a581753f149b49")
check("n_int_ur_2026", n_int_ur_2026, "9ce5e3435e75fc36")
check("n_int_both_2026", n_int_both_2026, "913ba482bdd2e764")
check("n_int_both_2021", n_int_both_2021, "8fa5d151b56bd8db")
print("ok  ex5")


**Write.** (a) Why can a *few* precincts land on 70.0 even when nobody touches the protocol, and why is a visible spike still a problem? Two sentences.

(b) Your `n_int_both_2026` is a raw count on a later, larger file. Why is it not Meduza's 2,250, and what would you still need in order to call it an *excess*? Two sentences.

YOUR ANSWER HERE


## Before you submit

- Restart the kernel and run every cell. Each test cell prints `ok`.
- Every YOUR PREDICTION HERE and YOUR ANSWER HERE is replaced.
- Cheat sheet check. Without looking, can you write: turnout and a party share from a protocol; what a positive slope of share on turnout would mean if extra ballots were independent of the party; why a vertical strip at low turnout is not the same as a diagonal tail; why honest integer vote counts can hit 50.0 sometimes, and why a pile at 70.0 is a different story? Whatever you cannot write goes on the sheet.
